# 02. 대상지 정의 · 격자 태깅

격자 54,029개(30m) 각각에 `zoneName`(4개 지구) + `blockType`(78종)을 부여한다.
한 격자가 여러 용도 폴리곤에 걸치므로 **면적 최대 중첩(area-majority)** 규칙 사용.

산출물: `data/interim/grid_tagged.geojson`, `output/figures/fig02_target_area.png`

In [1]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
import geopandas as gpd
from shapely import force_2d
from src.config import GRID_30M, LANDUSE, DATA_INTERIM, OUT_FIGURES, CRS_INTERNAL, CRS_SUBMIT, ZONE_COLORS

grid = gpd.read_file(GRID_30M)[["grid_id", "geometry"]]
lu = gpd.read_file(LANDUSE)[["zoneName", "blockType", "geometry"]].copy()
lu["geometry"] = force_2d(lu.geometry)   # Z 좌표 제거
grid.shape, lu.shape

((54029, 2), (3832, 3))

In [2]:
# 면적 최대 중첩 태깅: 후보쌍(sjoin) → 교차면적(미터) → 격자별 최대면적 용도 선택
# reset_index 필수: sjoin은 왼쪽 인덱스를 중복 보존 → .loc[idxmax]가 중복행 반환
pairs = gpd.sjoin(grid, lu, predicate="intersects", how="inner").reset_index(drop=True)

grid_m = grid.to_crs(CRS_INTERNAL).set_index("grid_id").geometry
lu_m = lu.to_crs(CRS_INTERNAL).geometry
lg = grid_m.loc[pairs["grid_id"].values].reset_index(drop=True)
rg = lu_m.loc[pairs["index_right"].values].reset_index(drop=True)
pairs["area"] = lg.intersection(rg, align=False).area.values

dom = pairs.loc[pairs.groupby("grid_id")["area"].idxmax(), ["grid_id", "zoneName", "blockType"]]
assert dom["grid_id"].is_unique, "grid_id 중복 — 태깅 로직 오류"
grid_tagged = grid.merge(dom, on="grid_id", how="left")
grid_tagged.shape

(54029, 4)

In [3]:
# 요약
n_tag = grid_tagged["zoneName"].notna().sum()
print(f"전체 {len(grid_tagged):,} / 태깅됨 {n_tag:,} / 미태깅(지구 밖) {len(grid_tagged)-n_tag:,}")
print(grid_tagged["zoneName"].value_counts(dropna=False))
print("\nblockType 상위:")
print(grid_tagged["blockType"].value_counts().head(10))

전체 54,029 / 태깅됨 54,029 / 미태깅(지구 밖) 0
zoneName
성남분당             21847
남양주왕숙 공공주택지구     12122
성남판교지구 택지개발사업    10277
고양창릉 공공주택지구       9783
Name: count, dtype: int64

blockType 상위:
blockType
도로      10102
아파트      7111
근린공원     4855
공원       4385
공동주택     4174
녹지       3524
하천       2951
단독주택     2334
업무시설     1445
연립       1229
Name: count, dtype: int64


In [4]:
# 저장 (EPSG:4326) + 지구별 색상 확인 지도
DATA_INTERIM.mkdir(parents=True, exist_ok=True)
OUT_FIGURES.mkdir(parents=True, exist_ok=True)
grid_tagged.to_crs(CRS_SUBMIT).to_file(DATA_INTERIM / "grid_tagged.geojson", driver="GeoJSON")

import matplotlib.pyplot as plt
from src.viz.fonts import setup_font
setup_font()   # 한글 폰트 자동 적용(COMPAS 미보유 시 동봉 폰트 등록)
ax = grid_tagged.plot(
    figsize=(9, 9), linewidth=0,
    color=grid_tagged["zoneName"].map(ZONE_COLORS).fillna("#dddddd"),
)
ax.set_axis_off(); ax.set_title("격자 태깅: 4개 지구")
plt.tight_layout()
plt.savefig(OUT_FIGURES / "fig02_target_area.png", dpi=130)
plt.close()